In [0]:
%sql
USE CATALOG nasa_dw;

CREATE TABLE IF NOT EXISTS gold.fact_acercamientos
USING DELTA
COMMENT 'Tabla de hechos unificada: asteroides y cometas'
AS
-- Crear estructura vacía leyendo desde silver y dimensiones
SELECT
    f.fecha_id,
    a.asteroide_id              AS cuerpo_id,
    'Asteroide'                 AS tipo_cuerpo,
    s.distancia_tierra_km,
    s.distancia_tierra_lu,
    s.velocidad_kmh,
    s.velocidad_kms,
    s.cuerpo_orbitado,
    CAST(NULL AS INT)           AS clasificacion_id,
    CASE
        WHEN s.distancia_tierra_lu < 1.0  THEN 4
        WHEN s.distancia_tierra_lu < 2.0  THEN 3
        WHEN s.distancia_tierra_lu < 5.0  THEN 2
        ELSE 1
    END                         AS impacto_id,
    CURRENT_TIMESTAMP()         AS fecha_procesamiento
FROM nasa_dw.silver.neo_clean   s
JOIN nasa_dw.gold.dim_asteroide a ON a.asteroide_id = s.asteroide_id
JOIN nasa_dw.gold.dim_fecha     f ON f.fecha = s.fecha_acercamiento
WHERE 1=0;

In [0]:
%sql
MERGE INTO nasa_dw.gold.fact_acercamientos AS tgt
USING (
    -- Asteroides
    SELECT
        f.fecha_id,
        a.asteroide_id              AS cuerpo_id,
        'Asteroide'                 AS tipo_cuerpo,
        s.distancia_tierra_km,
        s.distancia_tierra_lu,
        s.velocidad_kmh,
        s.velocidad_kms,
        s.cuerpo_orbitado,
        CAST(NULL AS INT)           AS clasificacion_id,
        CASE
            WHEN s.distancia_tierra_lu < 1.0  THEN 4
            WHEN s.distancia_tierra_lu < 2.0  THEN 3
            WHEN s.distancia_tierra_lu < 5.0  THEN 2
            ELSE 1
        END                         AS impacto_id,
        CURRENT_TIMESTAMP()         AS fecha_procesamiento
    FROM nasa_dw.silver.neo_clean   s
    JOIN nasa_dw.gold.dim_asteroide a ON a.asteroide_id = s.asteroide_id
    JOIN nasa_dw.gold.dim_fecha     f ON f.fecha = s.fecha_acercamiento
    
    UNION ALL
    
    -- Cometas
    SELECT
        f.fecha_id,
        c.cometa_id                 AS cuerpo_id,
        'Cometa'                    AS tipo_cuerpo,
        s.distancia_km              AS distancia_tierra_km,
        s.distancia_lu              AS distancia_tierra_lu,
        s.velocidad_kmh,
        s.velocidad_kms,
        'Sun'                       AS cuerpo_orbitado,
        CASE
            WHEN s.periodo = 'Corto' THEN 5
            WHEN s.periodo = 'Largo' THEN 6
            ELSE NULL
        END                         AS clasificacion_id,
        CASE
            WHEN s.distancia_lu < 1.0  THEN 4
            WHEN s.distancia_lu < 2.0  THEN 3
            WHEN s.distancia_lu < 5.0  THEN 2
            ELSE 1
        END                         AS impacto_id,
        CURRENT_TIMESTAMP()         AS fecha_procesamiento
    FROM nasa_dw.silver.cometas_clean s
    JOIN nasa_dw.gold.dim_cometa      c ON c.cometa_id = s.cometa_id
    JOIN nasa_dw.gold.dim_fecha       f ON f.fecha = s.fecha_acercamiento
) AS src
ON tgt.fecha_id = src.fecha_id
AND tgt.cuerpo_id = src.cuerpo_id
AND tgt.tipo_cuerpo = src.tipo_cuerpo
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
SELECT
    f.cuerpo_id,
    COALESCE(a.nombre, c.nombre)    AS nombre_astro,
    f.tipo_cuerpo,
    d.fecha                         AS fecha_acercamiento,
    f.distancia_tierra_km,
    f.velocidad_kmh
FROM nasa_dw.gold.fact_acercamientos f
JOIN nasa_dw.gold.dim_fecha          d ON d.fecha_id = f.fecha_id
LEFT JOIN nasa_dw.gold.dim_asteroide a ON a.asteroide_id = f.cuerpo_id AND f.tipo_cuerpo = 'Asteroide'
LEFT JOIN nasa_dw.gold.dim_cometa    c ON c.cometa_id = f.cuerpo_id AND f.tipo_cuerpo = 'Cometa'
ORDER BY f.distancia_tierra_km ASC
LIMIT 20;

cuerpo_id,nombre_astro,tipo_cuerpo,fecha_acercamiento,distancia_tierra_km,velocidad_kmh
54554706,(2025 US6),Asteroide,2025-11-13,136610.52,7480.12
54554706,(2025 US6),Asteroide,2025-09-10,150841.66,6997.91
54554706,(2025 US6),Asteroide,2025-10-28,151088.54,7021.94
54554706,(2025 US6),Asteroide,2025-10-12,154424.09,6919.53
54554706,(2025 US6),Asteroide,2025-12-02,159495.55,6814.32
54554706,(2025 US6),Asteroide,2025-09-26,160725.8,6730.55
3634154,(2013 GM3),Asteroide,2026-04-14,260528.06,26674.23
54441093,(2024 JV8),Asteroide,2024-02-23,364242.69,5342.68
54554706,(2025 US6),Asteroide,2025-09-03,381125.23,5629.5
54554706,(2025 US6),Asteroide,2025-11-22,382219.66,922.88


In [0]:
%sql
select * from nasa_dw.gold.fact_acercamientos;

fecha_id,cuerpo_id,tipo_cuerpo,distancia_tierra_km,distancia_tierra_lu,velocidad_kmh,velocidad_kms,cuerpo_orbitado,fecha_procesamiento,clasificacion_id,impacto_id
20251130,2000433,Asteroide,5.948721518E7,154.7534,13423.86,3.7289,Earth,2026-07-21T23:38:09.114Z,null,1
20250108,2000887,Asteroide,1.229660509E7,31.9891,29695.32,8.2487,Earth,2026-07-21T23:38:09.114Z,null,1
20240120,2001685,Asteroide,1.990593487E7,51.7844,58226.72,16.1741,Earth,2026-07-21T23:38:09.114Z,null,1
20260530,2001943,Asteroide,1.977135766E7,51.4343,26606.92,7.3908,Earth,2026-07-21T23:38:09.114Z,null,1
20240331,2002063,Asteroide,1.795252541E7,46.7027,30698.84,8.5275,Earth,2026-07-21T23:38:09.114Z,null,1
20250812,2002100,Asteroide,2.586603424E7,67.2894,44009.56,12.2249,Earth,2026-07-21T23:38:09.114Z,null,1
20241102,2002340,Asteroide,5.948137446E7,154.7382,93669.38,26.0193,Earth,2026-07-21T23:38:09.114Z,null,1
20260117,2002340,Asteroide,1.561912694E7,40.6325,50090.04,13.9139,Earth,2026-07-21T23:38:09.114Z,null,1
20251119,2003361,Asteroide,5673381.53,14.7591,32676.65,9.0768,Earth,2026-07-21T23:38:09.114Z,null,1
20250401,2004034,Asteroide,2.345059607E7,61.0057,43128.81,11.9802,Earth,2026-07-21T23:38:09.114Z,null,1


In [0]:
%sql
SELECT
    i.nivel                                  AS nivel_impacto,
    f.tipo_cuerpo,
    COUNT(*)                                 AS cantidad,
    ROUND(AVG(f.distancia_tierra_lu), 2)     AS dist_promedio_lu,
    ROUND(AVG(f.velocidad_kmh), 0)           AS velocidad_promedio_kmh
FROM nasa_dw.gold.fact_acercamientos f
JOIN nasa_dw.gold.dim_impacto        i ON i.impacto_id = f.impacto_id
GROUP BY i.nivel, f.tipo_cuerpo
ORDER BY i.nivel, f.tipo_cuerpo;

nivel_impacto,tipo_cuerpo,cantidad,dist_promedio_lu
Alto,Asteroide,7,1.63
Bajo,Asteroide,4623,103.47
Bajo,Cometa,3,79.99
Crítico,Asteroide,10,0.6
Moderado,Asteroide,20,3.77
